# 02 — Training Model & Backtest

Pipeline:
1. Bangun fitur per pertandingan (Elo pra-laga, form rolling, head-to-head — **bebas leakage**)
2. **Backtest walk-forward** pada Piala Dunia 2018 & 2022: train hanya dengan data sebelum turnamen
3. Bandingkan vs baseline (logistik Elo-only & prior frekuensi)
4. Latih model final pada seluruh data < 11 Juni 2026, simpan ke `models/` + `output/metrics.json`

**Arsitektur model — ensemble 4 komponen:**
- **XGBoost Poisson** → expected goals (λ) kedua tim → prediksi skor + probabilitas turunan grid Poisson
- **XGBoost classifier W/D/L** terkalibrasi isotonic
- **Elo-logistik** — regularisasi maksimal, paling stabil di laga Piala Dunia
- **Prior frekuensi** — shrinkage ke base-rate

Bobot kombinasi di-tuning via grid search dengan **validasi bersarang pada 2 edisi Piala Dunia sebelumnya** (untuk model 2026: divalidasi pada PD 2018 & 2022) — sehingga bobot dioptimalkan persis untuk distribusi laga Piala Dunia, tanpa menyentuh data tes. Sampel training diberi bobot time-decay (half-life 10 tahun).

In [1]:
import sys
from pathlib import Path

BASE = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(BASE / 'src'))

import numpy as np
import pandas as pd
import plotly.express as px

from data_prep import load_results, TOURNAMENT_START
from features import build_match_features, FEATURE_COLS
import models as M

results = load_results()
featured = build_match_features(results)
print(f'{len(featured):,} pertandingan dengan {len(FEATURE_COLS)} fitur siap.')
featured[FEATURE_COLS + ['home_score', 'away_score', 'outcome']].tail(3)

49,405 pertandingan dengan 15 fitur siap.


,elo_home,elo_away,elo_diff,form5_home,form5_away,form10_home,form10_away,gf10_home,ga10_home,gf10_away,ga10_away,h2h_winrate_home,h2h_matches,neutral,importance,home_score,away_score,outcome
49402,2042.643932,1886.047783,256.596149,2.0,2.2,2.1,2.2,2.6,1.0,2.0,0.6,1.000000,1.0,0.0,20.0,2,1,0
49403,1728.672248,1873.092624,-144.420375,1.2,2.0,1.0,2.3,0.8,1.7,1.9,0.4,0.000000,1.0,1.0,20.0,0,4,2
49404,1201.199936,1092.200078,108.999858,1.0,1.6,0.5,0.8,0.4,1.4,0.6,2.2,0.333333,9.0,1.0,30.0,0,2,2


## Backtest Piala Dunia 2018 & 2022

In [2]:
bt = {}
for year in (2018, 2022):
    bt[year] = M.backtest_world_cup(featured, year)
    print(f'PD {year}: {bt[year]["n_matches"]} laga | bobot ensemble (clf, poisson, elo, prior) = {bt[year]["blend_weights"]}')

rows = []
for year, res in bt.items():
    for name in ['model_blend', 'model_classifier', 'model_poisson', 'baseline_elo_logistic', 'baseline_prior']:
        rows.append({'PD': year, 'model': name, **res[name]})
bt_df = pd.DataFrame(rows).set_index(['PD', 'model']).round(4)
bt_df

PD 2018: 64 laga | bobot ensemble (clf, poisson, elo, prior) = [np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)]
PD 2022: 64 laga | bobot ensemble (clf, poisson, elo, prior) = [np.float64(0.0), np.float64(0.0), np.float64(1.0), np.float64(0.0)]


log_loss   brier  accuracy
PD   model                                            
2018 model_blend              0.9834  0.5847    0.5469
     model_classifier         1.0187  0.6073    0.5469
     model_poisson            1.0242  0.6103    0.4531
     baseline_elo_logistic    0.9834  0.5847    0.5469
     baseline_prior           1.0941  0.6678    0.3906
2022 model_blend              1.0745  0.6255    0.4844
     model_classifier         1.1131  0.6388    0.5156
     model_poisson            1.1088  0.6461    0.5156
     baseline_elo_logistic    1.0745  0.6255    0.4844
     baseline_prior           1.0743  0.6512    0.4375

In [3]:
plot_df = bt_df.reset_index()
fig = px.bar(plot_df, x='model', y='log_loss', color='PD', barmode='group',
             title='Log-loss per model (makin rendah makin baik)')
fig.show()
for year in (2018, 2022):
    blend = bt[year]['model_blend']
    base = bt[year]['baseline_elo_logistic']
    assert blend['log_loss'] <= base['log_loss'] + 0.02, f'Model kalah jauh dari baseline di {year}!'
    print(f"PD {year}: log-loss blend {blend['log_loss']:.4f} vs baseline Elo {base['log_loss']:.4f} | akurasi {blend['accuracy']:.1%} | MAE gol {bt[year]['goals_mae']['home']:.2f}/{bt[year]['goals_mae']['away']:.2f}")

PD 2018: log-loss blend 0.9834 vs baseline Elo 0.9834 | akurasi 54.7% | MAE gol 0.96/0.78
PD 2022: log-loss blend 1.0745 vs baseline Elo 1.0745 | akurasi 48.4% | MAE gol 1.14/0.89


## Kurva kalibrasi (PD 2022)
Probabilitas prediksi vs frekuensi aktual — garis diagonal = kalibrasi sempurna.

In [4]:
test22 = featured[(featured.tournament == 'FIFA World Cup') & (featured.date.dt.year == 2022)]
cutoff22 = test22.date.min()
train22 = featured[featured.date < cutoff22]
w22 = M.tune_blend_weights(train22, cutoff22)
mdl22 = M.train_models(train22, cutoff22)
probs22 = M.ensemble_probs(mdl22, test22[FEATURE_COLS].to_numpy(), weights=w22)

flat_p = probs22.ravel()
flat_y = np.eye(3)[test22.outcome.to_numpy()].ravel()
bins = np.linspace(0, 1, 6)
ix = np.digitize(flat_p, bins) - 1
cal = pd.DataFrame({'prediksi': [flat_p[ix == b].mean() for b in range(5)],
                    'aktual': [flat_y[ix == b].mean() for b in range(5)]}).dropna()
fig = px.scatter(cal, x='prediksi', y='aktual', title='Kalibrasi probabilitas — PD 2022')
fig.add_shape(type='line', x0=0, y0=0, x1=1, y1=1, line=dict(dash='dash'))
fig.show()

## Training final & feature importance
Model final dilatih pada seluruh data sebelum kickoff 11 Juni 2026; bobot ensemble divalidasi pada PD 2018 & 2022.

In [5]:
final_models = M.run_full_training(featured, TOURNAMENT_START)
print('Bobot ensemble final (clf, poisson, elo, prior):', final_models['weights'])
imp = pd.Series(final_models['goals_home'].feature_importances_, index=FEATURE_COLS).sort_values()
fig = px.bar(imp, orientation='h', title='Feature importance — model gol kandang (gain)',
             labels={'index': '', 'value': 'importance'}).update_layout(showlegend=False)
fig.show()

metrics = {str(y): bt[y] for y in bt}
metrics['final_blend_weights'] = list(final_models['weights'])
metrics['feature_importance'] = imp.round(4).to_dict()
path = M.save_metrics(metrics)
print(f'Model tersimpan di models/wc2026_models.joblib; metrik di {path}')

Bobot ensemble final (clf, poisson, elo, prior): (np.float64(0.0), np.float64(0.0), np.float64(0.8), np.float64(0.2))


Model tersimpan di models/wc2026_models.joblib; metrik di E:\WorldCup-Predictive\output\metrics.json
